In [ ]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Dataset downloaded to:", path)

# Find the CSV file (usually named 'spam.csv' or similar)
for file in os.listdir(path):
    if file.endswith('.csv'):
        csv_path = os.path.join(path, file)
        print("CSV file found:", csv_path)
        break

# Load the CSV
df = pd.read_csv(csv_path, encoding='latin-1')
print("First 5 rows:")
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import kagglehub
import os

nltk.download('stopwords', quiet=True)

In [ ]:
# ------------------------------------------------------------
# 1. Load the SMS Spam Collection dataset
# ------------------------------------------------------------
print("Step 1: Downloading dataset...")
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Dataset path:", path)

# Locate the CSV file
csv_file = None
for file in os.listdir(path):
    if file.endswith('.csv'):
        csv_file = os.path.join(path, file)
        break

if csv_file is None:
    raise FileNotFoundError("No CSV file found in the downloaded dataset")

print("Loading CSV:", csv_file)
df = pd.read_csv(csv_file, encoding='latin-1')

# The SMS dataset has columns: v1 (label), v2 (message)
# Drop any extra unnamed columns
df = df[['v1', 'v2']]
df.columns = ['label', 'text']   # rename to match our code

# Convert labels: spam -> 1, ham -> 0
df['label'] = df['label'].map({'spam': 1, 'ham': 0})

print(f"Total samples: {len(df)}")
print(f"Spam count: {df['label'].sum()} | Ham count: {len(df) - df['label'].sum()}")
print("Sample data:")
print(df.head())


In [ ]:
# ------------------------------------------------------------
# 2. Text preprocessing
# ------------------------------------------------------------

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

X = df['text'].apply(preprocess_text)
y = df['label']

In [ ]:
# ------------------------------------------------------------
# 3. Train-test split
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


In [ ]:
# ------------------------------------------------------------
# 4. TF-IDF vectorization
# ------------------------------------------------------------
print("Step 3: Vectorizing with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf)
print(X_test_tfidf)


In [ ]:
# ------------------------------------------------------------
# 5. Train Naive Bayes classifier
# ------------------------------------------------------------
print(" Training Multinomial Naive Bayes...")
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)


In [ ]:
# ------------------------------------------------------------
# 6. Evaluate
# ------------------------------------------------------------
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n=== Evaluation Results ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)


In [ ]:
# ------------------------------------------------------------
# 7. Save model and vectorizer
# ------------------------------------------------------------
joblib.dump(model, 'spam_detector_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("\nModel and vectorizer saved successfully.")

In [ ]:
# ------------------------------------------------------------
# 8. Prediction example
# ------------------------------------------------------------
def predict_email(email_text):
    model = joblib.load('spam_detector_model.pkl')
    tfidf = joblib.load('tfidf_vectorizer.pkl')
    cleaned = preprocess_text(email_text)
    vec = tfidf.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    return pred, prob

new_email = """
Congratulations! You've won a $1000 Amazon gift card.
Click here to claim your prize now: http://fake-link.com
"""
pred, prob = predict_email(new_email)
print("\n--- Example Prediction ---")
print(f"Email text: {new_email[:100]}...")
print(f"Prediction: {'SPAM' if pred == 1 else 'HAM'} (probability: {prob[pred]:.2f})")